# Superconductor Tc Extraction — Qwen VLM Pipeline Results

Clean visualisation notebook for the HF-dataset-driven pipeline
(`run_from_hf.py`): materials + synthesis method come from
`structured_synthesis` (no LLM re-extraction), Tc comes from a single
Qwen VLM call per figure (`tc_vlm` column — no `_orig`/`_snip` split,
no separate digitization/linking pass).

Point `RESULTS_DIR` at whichever run you want to inspect:
- `results_superconductors_hf_snippet19` — 19 papers with human/text Tc
  ground truth already in `results/results_superconductors/tc_master_snippet.csv`
  (panel b comparison)
- `results_superconductors_hf_100` — 100-paper trial (panels c/d)

In [ ]:
import sys
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

In [ ]:
FIGSIZE = (4, 4)

# --- switch which run to inspect here ---
RESULTS_DIR = Path(
    "/home/magled/lematerial-llm-synthesis/data/results_superconductors_hf_snippet19"
)
# RESULTS_DIR = Path(
#     "/home/magled/lematerial-llm-synthesis/data/results_superconductors_hf_100"
# )

# Old Claude-based run: has tc_text (human/text-extracted ground truth) +
# tc_vlm_orig (Claude Sonnet 4.6). Used ONLY for the panel-b ground-truth
# join -- not merged wholesale like the old notebook did.
GROUND_TRUTH_CSV = Path(
    "/home/magled/lematerial-llm-synthesis/results/results_superconductors/tc_master_snippet.csv"
)

In [ ]:
SRC_DIR = str(Path().resolve().parents[2] / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from llm_synthesis.utils.style_utils import set_style, get_palette

set_style("presentation")
sns.set_style("white")

PAL = get_palette()
C_BLUE = PAL[2]
C_ORANGE = PAL[4]
C_PURPLE = PAL[12]
C_PINK = PAL[0]
C_GREY = PAL[8]
C_DGREY = PAL[10]


def set_pub_style(style: str = "manuscript") -> None:
    """Port of icicle.utils.visualization.style.set_style (typography/axes
    only -- color cycle uses this notebook's PAL)."""
    sz = {
        "manuscript": {
            "font": 10,
            "label": 10,
            "title": 10,
            "tick": 9,
            "legend": 9,
            "major_tick": 3,
        },
        "presentation": {
            "font": 12,
            "label": 12,
            "title": 12,
            "tick": 11,
            "legend": 11,
            "major_tick": 4,
        },
    }[style]
    settings = {
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "mathtext.fontset": "custom",
        "mathtext.rm": "Arial",
        "mathtext.it": "Arial:italic",
        "mathtext.bf": "Arial:bold",
        "mathtext.sf": "Arial",
        "font.size": sz["font"],
        "axes.labelsize": sz["label"],
        "axes.titlesize": sz["title"],
        "xtick.labelsize": sz["tick"],
        "ytick.labelsize": sz["tick"],
        "legend.fontsize": sz["legend"],
        "legend.title_fontsize": sz["legend"],
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "figure.facecolor": "white",
        "axes.spines.top": True,
        "axes.spines.right": True,
        "axes.linewidth": 1.4,
        "axes.edgecolor": "black",
        "axes.labelcolor": "black",
        "axes.grid": False,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": sz["major_tick"],
        "ytick.major.size": sz["major_tick"],
        "legend.frameon": False,
        "axes.prop_cycle": plt.cycler("color", PAL[:8]),
    }
    for k, v in settings.items():
        mpl.rcParams[k] = v


set_pub_style("manuscript")

## Load

Single CSV, single schema (`tc_vlm`, `synthesis_method` both present --
no merging two separate datasets like the old notebook).

In [ ]:
df = pd.read_csv(RESULTS_DIR / "tc_master.csv")
print(f"{len(df)} rows, {df['paper_id'].nunique()} papers")
print("columns:", list(df.columns))
df.head()

In [ ]:
def classify_family(mat: str) -> str:
    mat_lower = str(mat).lower()
    if (
        "mgb" in mat_lower
        or "alb" in mat_lower
        or re.search(r"\w+b2", mat_lower)
    ):
        return "Borides (MgB$_2$-type)"
    if "feas" in mat_lower or "feaso" in mat_lower:
        return "Iron pnictides"
    if (
        "fese" in mat_lower
        or "fete" in mat_lower
        or re.search(r"fe.*te.*se", mat_lower)
        or re.search(r"fe.*se", mat_lower)
    ):
        return "Iron chalcogenides"
    if "cu" in mat_lower and any(
        x in mat_lower for x in ("ba", "sr", "la", "ca")
    ):
        return "Cuprates"
    if "bis" in mat_lower or "bise" in mat_lower:
        return "BiS$_2$-based"
    if "re" in mat_lower and "mo" in mat_lower:
        return "Re-Mo alloys"
    if "hote" in mat_lower or ("pd" in mat_lower and "te" in mat_lower):
        return "Tellurides"
    if "ta" in mat_lower and "pd" in mat_lower and "s" in mat_lower:
        return "Chalcogenides (Ta$_2$PdS$_5$)"
    return "Other"


df["family"] = df["material_normalized"].apply(classify_family)

# year is already derived from arxiv id in CsvMasterWriter -- just clean types
for col in ("has_text_tc", "has_vlm_tc"):
    if col in df.columns:
        df[col] = df[col].astype(bool)

sc = df[df["is_superconductor"] == True].copy()
print(f"Superconductors: {len(sc)} / {len(df)}")
print(f"Year range: {df['year'].min()}–{df['year'].max()}")
print(f"Families: {df['family'].value_counts().to_dict()}")

## Panel b — Tc (Qwen VLM) vs Tc (human/text)

Ground truth lives in the OLD run's snippet CSV
(`results/results_superconductors/tc_master_snippet.csv`, column
`tc_text`) — join on `(paper_id, material)`, don't merge whole dataframes.

In [ ]:
gt = pd.read_csv(GROUND_TRUTH_CSV)[
    ["paper_id", "material_normalized", "tc_text"]
].copy()
# old CSV's paper_id has a "_<id>v<N>" suffix (e.g. "0801.2629_0801.2629v2");
# our new pipeline's paper_id is the bare arxiv id ("0801.2629") -- normalise
# by taking the part before the first underscore, and old-style ids
# (cond-mat/...) have their "/" already turned into "_" on our side.
gt["paper_id_norm"] = gt["paper_id"].str.split("_").str[0]

sc["paper_id"] = sc["paper_id"].astype(str)

cmp_df = sc.merge(
    gt[["paper_id_norm", "material_normalized", "tc_text"]],
    left_on=["paper_id", "material_normalized"],
    right_on=["paper_id_norm", "material_normalized"],
    how="inner",
)
print(f"Matched {len(cmp_df)} (paper, material) rows against ground truth")
cmp_df[["paper_id", "material", "tc_vlm", "tc_text"]].head(10)

In [ ]:
both = cmp_df.dropna(subset=["tc_text", "tc_vlm"]).copy()

if len(both) < 3:
    print(
        f"Only {len(both)} matched rows with both tc_text and tc_vlm -- "
        "not enough to plot yet. Re-check RESULTS_DIR / GROUND_TRUTH_CSV, "
        "or wait for the run to finish."
    )
else:
    max_tc = max(both["tc_text"].max(), both["tc_vlm"].max()) * 1.1
    fig, ax = plt.subplots(figsize=FIGSIZE)
    sns.regplot(
        x="tc_text",
        y="tc_vlm",
        data=both,
        ci=95,
        scatter_kws={
            "s": 45,
            "edgecolors": "k",
            "linewidths": 0.3,
            "zorder": 3,
            "color": C_BLUE,
        },
        line_kws={"lw": 1.5, "color": C_PURPLE},
        ax=ax,
    )
    ax.plot([0, max_tc], [0, max_tc], "k--", lw=1, label="$y = x$")
    r2 = both[["tc_text", "tc_vlm"]].corr().iloc[0, 1] ** 2
    mae = (both["tc_text"] - both["tc_vlm"]).abs().mean()
    ax.text(
        0.05,
        0.92,
        f"$R^2$ = {r2:.3f}\nMAE = {mae:.1f} K\n$n$ = {len(both)}",
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=1.0),
    )
    ax.set_xlim(-4, max_tc)
    ax.set_ylim(-4, max_tc)
    ax.set_xlabel("$T_c$ from text (human, K)")
    ax.set_ylabel("$T_c$ from VLM (Qwen, K)")
    ax.set_title("Qwen VLM vs Human Text $T_c$")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
    print(f"R²={r2:.3f}  MAE={mae:.2f} K  n={len(both)}")

## Panel c — Materials × synthesis method → Tc

`synthesis_method` is already in `df`/`sc` (from `structured_synthesis`,
no separate dataset load needed).

In [ ]:
plot_df = sc.dropna(subset=["tc_best", "synthesis_method"]).copy()
plot_df = plot_df[plot_df["synthesis_method"].str.strip() != ""]
print(f"{len(plot_df)} rows with both Tc and synthesis_method")

if len(plot_df) < 3:
    print(
        "Not enough rows yet -- re-check RESULTS_DIR or wait for the run to finish."
    )
else:
    order = plot_df.groupby("family")["tc_best"].median().sort_values().index
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.boxplot(
        data=plot_df,
        x="tc_best",
        y="family",
        order=order,
        color=C_GREY,
        fliersize=0,
        ax=ax,
    )
    sns.stripplot(
        data=plot_df,
        x="tc_best",
        y="family",
        order=order,
        hue="synthesis_method",
        size=5,
        alpha=0.8,
        edgecolor="k",
        linewidth=0.3,
        ax=ax,
        legend="brief",
    )
    ax.set_xlabel("$T_c$ (K)")
    ax.set_ylabel("")
    ax.legend(
        fontsize=7,
        title="Synthesis method",
        title_fontsize=8,
        loc="lower right",
    )
    plt.tight_layout()
    plt.show()

## Panel d — Tc vs year (scale-up demonstration)

In [ ]:
plot_yr = sc.dropna(subset=["tc_best", "year"]).copy()
print(f"{len(plot_yr)} rows with both Tc and year")

if len(plot_yr) < 3:
    print(
        "Not enough rows yet -- re-check RESULTS_DIR or wait for the run to finish."
    )
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(
        data=plot_yr,
        x="year",
        y="tc_best",
        hue="family",
        s=60,
        alpha=0.85,
        edgecolor="k",
        linewidth=0.3,
        ax=ax,
    )
    ax.axhline(
        77,
        color="red",
        linestyle="--",
        lw=1,
        alpha=0.6,
        label="Boiling point of N$_2$",
    )
    ax.set_xlabel("Year")
    ax.set_ylabel("$T_c$ (K)")
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.tight_layout()
    plt.show()